<a href="https://colab.research.google.com/github/MbHashi/Flask-Inventory-System/blob/main/FraudDetectionScript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#Importing the necessary libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from imblearn.over_sampling import SMOTE


In [6]:

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Libraries
from sklearn.model_selection import train_test_split # Used to split the dataset into training and testing sets
from sklearn.preprocessing import StandardScaler # To scale the data
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report # Used to evaluate the performance
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Using the 'time' library to track training of the SVM model
import time

# Deep Learning Libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
print ("All libraries imported successfully!")
from scipy.stats import ks_2samp


All libraries imported successfully!


In [7]:
# #Importing the dataset
# df = pd.read_csv('loan.csv')
# print(f"Dataset loaded successfully with {df.shape[0]} rows.")
# Importing the dataset from my Google drive:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DATA5000 - Group Project/Group Project/loan.csv')

/tmp/ipython-input-8-3774817203.py:1: DtypeWarning: Columns (19,47,55,112,123,124,125,128,129,130,133,139,140,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/DATA5000 - Group Project/Group Project/loan.csv')


In [ ]:
df.head(10)  # Display the first 10 rows of the dataset


In [ ]:
'''Cleaning the dataset by first filtering the data to keep only
homeowners & mortgage.

Then we will continue to use the date range as per the paper'''

df = df[df['home_ownership'].isin(['MORTGAGE', 'OWN'])].copy()
print(f"Dataset now has {df.shape[0]} rows.")

In [ ]:
# We need to convert the 'issue_d' column to a datetime format first.
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df = df[(df['issue_d'] >= '2016-01-01') & (df['issue_d'] <= '2017-12-31')].copy()
print(f"Filtered data to {df.shape[0]} rows based on home ownership and date range.")


In [ ]:
# The Target and Clean Features:
# We still need to define what a "fraudulent" loan is for our model.
df_filtered = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df_filtered['is_fraudulent'] = df_filtered['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)
print("Target variable: 'is_fraudulent' created.")


In [ ]:
'''Creating visuals to better understand the data:'''
# Plot 1: Checking if the data is imbalanced:
# This plot will show us the distribution of our target variable.
plt.figure(figsize=(7, 5))
sns.countplot(x='is_fraudulent', data=df_filtered)
plt.title('Distribution of Fraudulent vs. Non-Fraudulent Loans')
plt.xticks([0, 1], ['Non-Fraudulent (0)', 'Fraudulent (1)'])
plt.ylabel('Number of Loans')
plt.show()
print("Note the imbalance: there are far more non-fraudulent loans than fraudulent ones.")
print(df_filtered['is_fraudulent'].value_counts(normalize=True))


In [ ]:
# Plot 2: How the loan amount differs for fraud vs. non-fraud:
# A histogram to show the distribution of key numerical features.
plt.figure(figsize=(10, 6))
sns.histplot(data=df_filtered, x='loan_amnt', hue='is_fraudulent', kde=True, bins=40)
plt.title('Distribution of Loan Amount by Fraud Status')
plt.xlabel('Loan Amount ($)')
plt.ylabel('Count')
plt.legend(title='Status', labels=['Fraudulent', 'Non-Fraudulent'])
plt.show()

In [ ]:
# Plot 3: How does loan purpose relate to fraud:
# A bar chart shows which loan purposes are most common for each class.
plt.figure(figsize=(12, 8))
sns.countplot(y='purpose', hue='is_fraudulent', data=df_filtered, order=df_filtered['purpose'].value_counts().index)
plt.title('Loan Purpose by Fraud Status')
plt.xlabel('Number of Loans')
plt.ylabel('Loan Purpose')
plt.tight_layout()
plt.show()

In [ ]:
''' Data Cleaning and Preprocessing:'''
# This is our curated list of columns to drop to prevent data leaks & remove noise.
columns_to_drop = [
    'id', 'member_id', 'grade', 'sub_grade', 'emp_title', 'issue_d',
    'loan_status', 'pymnt_plan', 'url', 'desc', 'title', 'zip_code',
    'addr_state', 'out_prncp', 'out_prncp_inv', 'total_pymnt',
    'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'policy_code',
    'debt_settlement_flag'
]

In [ ]:
# check which columns from our list actually exist in the df to avoid errors:
existing_cols_to_drop = [col for col in columns_to_drop if col in df_filtered.columns]
df_final = df_filtered.drop(columns=existing_cols_to_drop)
print(f"Removed {len(existing_cols_to_drop)} unnecessary columns.")


In [ ]:
# Handling text data with label encoding:
from sklearn.preprocessing import LabelEncoder
for col in df_final.select_dtypes(include=['object']).columns:
    df_final[col] = LabelEncoder().fit_transform(df_final[col].astype(str))
print("text-based columns converted to numbers using Label Encoding.")


In [ ]:
#handling missing values using the median::
df_final.fillna(df_final.mean(), inplace=True)


In [ ]:
# Defining features and target variable:
x = df_final.drop('is_fraudulent', axis=1)
y = df_final['is_fraudulent']


In [ ]:
# Looking to calculate how correlated the features are to the target variable:
correlations = x.corrwith(y).abs().sort_values(ascending=False)
top_features = correlations.head(30).index
x_featured = x[top_features]
print("Top 30 features selected based on correlation.")


In [ ]:
# Splitting the dataset into training and testing:
x_train, x_test, y_train, y_test = train_test_split(x_featured, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# Using SMOTE to balance the training data:
smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)
print("Training data balanced using SMOTE.")
print(f"Size of the new balanced training data: {x_train_resampled.shape}")


In [ ]:
# Scale the data so all features have a similar range:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_resampled)
x_test_scaled = scaler.transform(x_test)


In [ ]:
'''Model Training and Evaluation:'''
results_list = []


In [ ]:
#Test with traditional ML models first:
#Logistic Regression:
print("\n--- Training Logistic Regression ---")
# Creating the model
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
# Train the model
log_reg_model.fit(x_train_scaled, y_train_resampled)
# Get predicted probabilities
y_pred_proba_lr = log_reg_model.predict_proba(x_test_scaled)[:, 1]
# Calculate metrics:
acc_lr = accuracy_score(y_test, log_reg_model.predict(x_test_scaled))
auc_lr = roc_auc_score(y_test, y_pred_proba_lr)
ks_lr = ks_2samp(y_pred_proba_lr[y_test == 0], y_pred_proba_lr[y_test == 1]).statistic
# Saving results:
results_list.append({'Model': 'Logistic Regression', 'ACC': acc_lr, 'AUC': auc_lr, 'KS': ks_lr})
print(f"Logistic Regression - AUC: {auc_lr:.4f}, KS: {ks_lr:.4f}")


In [ ]:
#Decision Tree:
print("\n--- Training Decision Tree ---")
# Create model:
tree_model = DecisionTreeClassifier(random_state=42)
# Training the model:
tree_model.fit(x_train_scaled, y_train_resampled)
# Get predicted probabilities
y_pred_proba_tree = tree_model.predict_proba(x_test_scaled)[:, 1]
# Calculate metrics
acc_tree = accuracy_score(y_test, tree_model.predict(x_test_scaled))
auc_tree = roc_auc_score(y_test, y_pred_proba_tree)
ks_tree = ks_2samp(y_pred_proba_tree[y_test == 0], y_pred_proba_tree[y_test == 1]).statistic
# Save results
results_list.append({'Model': 'Decision Tree', 'ACC': acc_tree, 'AUC': auc_tree, 'KS': ks_tree})
print(f"Decision Tree - AUC: {auc_tree:.4f}, KS: {ks_tree:.4f}")


In [ ]:
#Random Forest:
print("\n--- Training Random Forest ---")
# Create the model:
rf_model = RandomForestClassifier(random_state=42)
# Train the model:
rf_model.fit(x_train_scaled, y_train_resampled)
# Get predicted probabilities:
y_pred_proba_rf = rf_model.predict_proba(x_test_scaled)[:, 1]
# Calculate metrics
acc_rf = accuracy_score(y_test, rf_model.predict(x_test_scaled))
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
ks_rf = ks_2samp(y_pred_proba_rf[y_test == 0], y_pred_proba_rf[y_test == 1]).statistic
# Save results
results_list.append({'Model': 'Random Forest', 'ACC': acc_rf, 'AUC': auc_rf, 'KS': ks_rf})
print(f"Random Forest - AUC: {auc_rf:.4f}, KS: {ks_rf:.4f}")


In [ ]:
#Support Vector Machine:]
print("\n- Training Support Vector Machine (SVM) -")
# NOTE: SVms can be very slow on large datasets, so we will train on a sample of the data.
n_samples_for_svm = 50000
print(f"SVM is slow, so we're training on a random sample of {n_samples_for_svm} data points.")
sample_indices = np.random.choice(x_train_scaled.shape[0], n_samples_for_svm, replace=False)
x_train_svm_sample = x_train_scaled[sample_indices]
y_train_svm_sample = y_train_resampled[sample_indices]

svm_model = SVC(probability=True, random_state=42)
# Time the model:
print("Starting SVM training... please be patient.")
start_time = time.time()
# Train the model on the smaller sample
svm_model.fit(x_train_svm_sample, y_train_svm_sample)
end_time = time.time()
duration = end_time - start_time
svm_model = SVC(probability=True, random_state=42)

# Time the model:
start_time = time.time()
# Train the model
svm_model.fit(x_train_scaled, y_train_resampled)
end_time = time.time()
duration = end_time - start_time
print(f"SVM training finished in {duration / 60:.2f} minutes.")
# Train the model
svm_model.fit(x_train_scaled, y_train_resampled)
# Get predicted probabilities
y_pred_proba_svm = svm_model.predict_proba(x_test_scaled)[:, 1]
# Calculate metrics
acc_svm = accuracy_score(y_test, svm_model.predict(x_test_scaled))
auc_svm = roc_auc_score(y_test, y_pred_proba_svm)
ks_svm = ks_2samp(y_pred_proba_svm[y_test == 0], y_pred_proba_svm[y_test == 1]).statistic
# Save results
results_list.append({'Model': 'Support Vector Machine', 'ACC': acc_svm, 'AUC': auc_svm, 'KS': ks_svm})
print(f"SVM - AUC: {auc_svm:.4f}, KS: {ks_svm:.4f}")

In [ ]:
'''Beginning to use Deep Learning models:'''
#%%
# Deep Neural Network (DNN):
# We build the model layer by layer.
dnn_model = Sequential([
    Dense(32, activation='relu', input_shape=(x_train_scaled.shape[1],)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid') # The final layer.
])


In [ ]:
dnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
'''Now we train it.
Note:'Epochs' are the many times the model sees the full dataset.'''
dnn_model.fit(x_train_scaled, y_train_resampled, epochs=10, batch_size=2048, verbose=0)
print("DNN training complete.")


In [ ]:
#We evaluate the trained DNN on our test data.
y_pred_proba_dnn = dnn_model.predict(x_test_scaled).ravel()
acc_dnn = accuracy_score(y_test, (y_pred_proba_dnn > 0.5).astype(int))
auc_dnn = roc_auc_score(y_test, y_pred_proba_dnn)
ks_dnn = ks_2samp(y_pred_proba_dnn[y_test == 0], y_pred_proba_dnn[y_test == 1]).statistic
# Save results
results_list.append({'Model': 'Deep Neural Network', 'ACC': acc_dnn, 'AUC': auc_dnn, 'KS': ks_dnn})
print(f"DNN - AUC: {auc_dnn:.4f}, KS: {ks_dnn:.4f}")


In [ ]:
'''Results Summary:'''
results_df = pd.DataFrame(results_list).set_index('Model')
results_df_sorted = results_df.sort_values(by='AUC', ascending=False)
print(results_df_sorted)


In [ ]:
#We can also plot the results to make it easier to see the differences:
results_df_sorted['AUC'].plot(kind='barh', figsize=(8, 5))
plt.title('Model Comparison by AUC Score')
plt.xlabel('AUC Score')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()